# Decoding the Rhythms of Emotion: A Sentimental Journey Through Music Genres
## Notebook 5 of 5 — Conclusion, References and Annexes

**Text Mining 2026/2027 | Group 15**

Raya Daher Alkhatib (20241416), Danyial Ahmad (20241831), Shaahir Racharla (20241419), Pedro Ambar (20231709)

| Notebook | Content |
|---|---|
| `01_eda` | Abstract, Introduction and Data Exploration |
| `02_preprocessing` | Initial Data Preprocessing |
| `03_genre_classification` | Genre Identification |
| `04_sentiment_analysis` | Sentiment Analysis |
| **`05_conclusion_annexes`** | **Conclusion, References and Annexes** |

**Run order:** `01 → 02 → 03 → 04 → 05`

This notebook closes the report. It has no new analysis: it reads the results saved by notebooks 3 and 4, so every number
quoted here is the one those notebooks printed.

| | |
|---|---|
| **Inputs** | `results/experiment_log.csv`, `results/submission_log.csv` (notebook 3), `results/sentiment_findings.csv` (notebook 4) |
| **Outputs** | — |
| **Expected runtime** | a few seconds |

In [1]:
%run ./_setup.py

for path in [config.EXPERIMENT_LOG, config.SUBMISSION_LOG, config.SENTIMENT_FINDINGS]:
    if not path.exists():
        raise FileNotFoundError(f"{path.name} is missing: run notebooks 03 and 04 first.")

experiments = pd.read_csv(config.EXPERIMENT_LOG)
submissions = pd.read_csv(config.SUBMISSION_LOG)
findings = pd.read_csv(config.SENTIMENT_FINDINGS)

Repo root: /home/claude/Music-Genre-Identification | seed: 42


## V. Conclusion

### 1. Summary of the findings

**Genre identification (notebook 3).** The final model and the file sent to Kaggle:

In [2]:
final = experiments.loc[experiments["exp_id"] == "03_final"]
final[["representation", "model", "params", "cv_macro_f1_mean", "cv_macro_f1_std", "notes"]].T

,0
representation,"TF-IDF word 1-2 (min_df=10) + char_wb 3-5 (min_df=10, max 250k)"
model,LinearSVC class_weight=balanced
params,C=0.1
cv_macro_f1_mean,0.5342
cv_macro_f1_std,0.0039
notes,"holdout macro F1 0.5410, accuracy 0.6555"


In [3]:
submissions[["file_name", "sha256", "cv_macro_f1", "public_lb_macro_f1", "notes"]]

,file_name,sha256,cv_macro_f1,public_lb_macro_f1,notes
0,Group15_Version02.csv,e66019a78149c1f66b519fe8b3cd487a67c65c85ee5d1be0a58e427483609b44,0.5342,0.551,"word (1,2) + char_wb (3,5) TF-IDF, balanced LinearSVC C=0.1, refit on all training songs"


**Can we tell a song's genre from its lyrics?** Partly. The final model, a class-balanced linear SVM on word and
character TF-IDF, reaches a macro F1 of **0.534 ± 0.004** in 5-fold cross-validation (0.541 on the 20% holdout, with
65.6% accuracy). That is far above the majority-class baseline (macro F1 0.097), but it is also far from perfect, and the
genres behave very differently (holdout F1 per genre, notebook 3):

- **rap** is recognised almost perfectly (0.867) and **misc** reasonably well (0.607). Both have a vocabulary and a
  structure of their own: slang and long verses for rap, old prose, plays and tables for misc.
- **pop** (0.663) is predicted well mainly because it is the largest genre.
- **rock** (0.469), **country** (0.346) and **rb** (0.295) are hard. Most of their mistakes are songs predicted as pop:
  pop has no vocabulary of its own, so it acts as the "default" genre for any song without strong genre markers.

On Kaggle's public leaderboard the submitted model scores **0.551**, slightly above the cross-validation estimate
(+0.017). The public leaderboard uses only 35% of the test songs, so a gap of this size is within normal variation; what
matters is that the estimate was not optimistic. Refitting on all training songs may also have helped a little.

Three steps made the difference on the holdout: balancing the class weights (+0.031 macro F1), tuning the regularisation
strength C (+0.022), and adding character n-grams (+0.009, mostly helping rb and country, the genres with the fewest
songs). Removing stopwords or the section-tag tokens each lowered the
holdout score by about 0.015, which confirmed the preprocessing choices of notebook 2. The submitted model uses the same
settings but is refitted on all 134,789 training songs; it agrees with the 80% model on 93.2% of the test songs.

**Sentiment analysis (notebook 4).** The main number behind each research question:

In [4]:
findings

,question,measure,value
0,Method check,"Spearman, VADER (per line) vs TextBlob, all songs",0.561
1,SQ1 genre,"most negative genre (rap), median vader_mean",-0.020
2,SQ1 genre,"most positive genre (country), median vader_mean",0.054
3,SQ1 genre,epsilon squared (share of sentiment variation explained by genre),0.065
4,SQ2 profanity,"Cliff's delta rap vs pop, VADER as is",-0.304
5,SQ2 profanity,"Cliff's delta rap vs pop, profanity neutral",-0.076
6,SQ3 time,"Spearman year vs sentiment, pop",-0.015
7,SQ3 time,"Spearman year vs sentiment, rap",-0.017
8,SQ3 time,"Spearman year vs sentiment, rock",-0.085
9,SQ3 time,"Spearman year vs sentiment, rb",-0.158


**How do the genres feel?** With VADER scored line by line on all 127,013 songs of the five music genres:

- **Country and rb are the most positive genres, rap and rock the least.** Rap is the only genre whose typical song is
  negative. The differences are real but modest: genre explains about 6.5% of the variation in sentiment (ε² = 0.065), and
  the largest gaps (rap vs country and rap vs rb) are medium effects.
- **Rap's negativity is mostly swearing.** Neutralising swear words and slurs, which VADER's lexicon scores as negative,
  shrinks the rap vs pop difference from a small effect (δ = -0.304) to a negligible one (δ = -0.076). After that, rock is
  the least positive genre.
- **Over time**, every genre became slightly more negative. The decline is steady in rb (ρ = -0.158), country and rock,
  and in pop and rap it is concentrated before 2000 (their correlations with year are below 0.02 in size).
- **Popularity** is practically unrelated to sentiment: all correlations with views are below 0.1 in absolute value, even
  when songs are only compared with songs from the same genre and year.
- **Within a song**, the chorus is slightly more positive than the verses in every genre (median difference about +0.01).

### 2. Did the results match what we expected?

The hypotheses come from the Introduction (notebook 1) and the objectives of notebook 4.

| Hypothesis | What we found | Verdict |
|---|---|---|
| Genres with a distinctive vocabulary (rap, country) are easy to recognise; pop, rock and rb are confused (Fell & Sporleder, 2014) | rap is by far the easiest genre (F1 0.867), but country is one of the hardest (0.346); pop, rock and rb are confused with each other, mostly towards pop | **Partly**: right for rap, wrong for country |
| Rap and rock are the most negative genres | they are the two lowest genres; rap is the only one with a negative median | **Supported** |
| Rap's negativity comes mostly from swear words, not from mood | rap vs pop goes from δ = -0.304 to δ = -0.076 once profanity is neutral | **Supported** |
| Lyrics became more negative over the decades (DeWall et al., 2011; Brand et al., 2019) | negative trend in every genre, but small (\|ρ\| ≤ 0.16) and, for pop and rap, mostly before 2000 | **Supported, weak effect** |
| Sentiment and popularity (no direction expected) | all \|ρ\| < 0.1 | no meaningful relation |
| The chorus is more positive than the verses | yes in every genre, but only by about 0.01 | **Supported, weak effect** |

The country result is the most interesting miss. Its vocabulary *is* distinctive (notebook 1 showed words like *honky
tonk* and *cowboy*), but most country songs don't use those words, and with only 2.6% of the songs the classifier sees too
few examples to learn the rest. A distinctive vocabulary helps only if most songs of the genre actually use it, which is
the case for rap and not for country.

The direction of the time trend matches DeWall et al. (2011) and Brand et al. (2019), but it is weak. Those studies followed chart hits year by
year; our data comes from a lyrics website, where old songs are a small and selected group (mostly classics people still
look up), so the comparison between decades is less clean.

### 3. Limitations

- **The genre labels are noisy.** Genres on the lyrics website are assigned roughly per artist, not per song, so a ballad by
  a rapper is labelled rap. Some mistakes of our classifier are probably mistakes in the labels.
- **We did not validate the sentiment scores against human judgement.** We checked the method in other ways (lyric-style
  test lines, whole-song vs line-level scores, agreement with TextBlob, reading the extreme songs), but we have no labelled
  lyrics, so we cannot say how *accurate* VADER is on songs, only that it is consistent and where it is biased.
- **Lexicons miss meaning that depends on context.** Slang ("this beat is sick"), irony and swearing used as emphasis
  are misread. The profanity test shows how large this effect can be for rap.
- **The notebooks are heavy to run.** Scoring every lyric line of 127,013 songs takes about 12 minutes, and refitting the
  final classifier on all songs needs about 6 GB of memory at its peak. We kept both because they make the results
  describe the whole dataset; on a machine with less than 8 GB of RAM the refit would have to fall back to the model
  trained on 80% of the songs (notebook 3, Section 7).
- **The data over-represents recent songs**, and the old songs that are present are not a random sample of their era.
  This limits the time analysis.
- **Views are not popularity in general.** They count visits to a lyrics page, which also depends on how hard the lyrics
  are to understand and on when the song was added to the site.
- **Only lyrics were used for classification.** We deliberately left out artist, year and views to answer the question in
  the brief (genre from lyrics), knowing that the artist name alone would raise the score.

### 4. Future work

- **Validate the sentiment method with human labels.** Two people labelling a few hundred lyric lines would let us measure
  VADER's accuracy on lyrics and Cohen's kappa between humans, which is the main gap in this work.
- **Adapt the lexicon to lyrics.** Re-scoring slang and swear words with their meaning in music ("sick", "killed it",
  in-group terms) would remove the biggest bias we found.
- **Go beyond positive and negative.** An emotion lexicon (joy, sadness, anger, fear) could separate, for example, the sad
  positivity of country from the angry negativity of rock, which a single polarity score mixes up.
- **Use structure and metadata in the classifier.** Rhyme, repetition and song-structure features (Mayer et al., 2008)
  and the artist could help the hard genres, especially rb and country.
- **Check the decade trends on chart data**, for example yearly top-100 lists, to remove the sampling bias of a lyrics
  website.

## References

Brand, C. O., Acerbi, A., & Mesoudi, A. (2019). Cultural evolution of emotional expression in 50 years of song lyrics.
*Evolutionary Human Sciences, 1*, e11. https://doi.org/10.1017/ehs.2019.11

Cliff, N. (1993). Dominance statistics: Ordinal analyses to answer ordinal questions. *Psychological Bulletin, 114*(3),
494–509. https://doi.org/10.1037/0033-2909.114.3.494

Cohen, J. (1960). A coefficient of agreement for nominal scales. *Educational and Psychological Measurement, 20*(1),
37–46. https://doi.org/10.1177/001316446002000104

Cortes, C., & Vapnik, V. (1995). Support-vector networks. *Machine Learning, 20*(3), 273–297.
https://doi.org/10.1007/BF00994018

DeWall, C. N., Pond, R. S., Jr., Campbell, W. K., & Twenge, J. M. (2011). Tuning in to psychological change: Linguistic
markers of psychological traits and emotions over time in popular U.S. song lyrics. *Psychology of Aesthetics,
Creativity, and the Arts, 5*(3), 200–207.

Efron, B., & Tibshirani, R. J. (1993). *An introduction to the bootstrap*. Chapman & Hall/CRC.

Fell, M., & Sporleder, C. (2014). Lyrics-based analysis and classification of music. In *Proceedings of COLING 2014,
the 25th International Conference on Computational Linguistics: Technical Papers*. Association for Computational
Linguistics. https://aclanthology.org/C14-1059/

Holm, S. (1979). A simple sequentially rejective multiple test procedure. *Scandinavian Journal of Statistics, 6*(2), 65–70.

Hutto, C. J., & Gilbert, E. (2014). VADER: A parsimonious rule-based model for sentiment analysis of social media text.
*Proceedings of the International AAAI Conference on Web and Social Media, 8*(1), 216–225.

Kruskal, W. H., & Wallis, W. A. (1952). Use of ranks in one-criterion variance analysis. *Journal of the American
Statistical Association, 47*(260), 583–621. https://doi.org/10.1080/01621459.1952.10483441

Mann, H. B., & Whitney, D. R. (1947). On a test of whether one of two random variables is stochastically larger than the
other. *The Annals of Mathematical Statistics, 18*(1), 50–60. https://doi.org/10.1214/aoms/1177730491

Mayer, R., Neumayer, R., & Rauber, A. (2008). Rhyme and style features for musical genre classification by song lyrics.
In *Proceedings of the 9th International Conference on Music Information Retrieval (ISMIR 2008)*.

Napier, K., & Shamir, L. (2018). Quantitative sentiment analysis of lyrics in popular music. *Journal of Popular Music
Studies, 30*(4), 161–176.

Romano, J., Kromrey, J. D., Coraggio, J., & Skowronek, J. (2006). Appropriate statistics for ordinal level data: Should
we really be using t-test and Cohen's d for evaluating group differences on the NSSE and other surveys? *Annual Meeting
of the Florida Association of Institutional Research*.

Spearman, C. (1904). The proof and measurement of association between two things. *The American Journal of Psychology,
15*(1), 72–101. https://doi.org/10.2307/1412159

Tomczak, M., & Tomczak, E. (2014). The need to report effect size estimates revisited. An overview of some recommended
measures of effect size. *Trends in Sport Sciences, 21*(1), 19–25.

Tsaptsinos, A. (2017). Lyrics-based music genre classification using a hierarchical attention network. In *Proceedings
of the 18th International Society for Music Information Retrieval Conference (ISMIR 2017)*.

Wilcoxon, F. (1945). Individual comparisons by ranking methods. *Biometrics Bulletin, 1*(6), 80–83.
https://doi.org/10.2307/3001968

## Annexes

The brief asks for theory only for methods that were **not** covered in the classes. TF-IDF, character n-grams,
Naive Bayes, Logistic Regression, the classification metrics, VADER and TextBlob were all covered, so they are not
repeated here. The methods below were used in notebooks 3 and 4 and come from outside the course.

### Annex A — Linear Support Vector Machine with class weights (notebook 3)

A linear SVM (Cortes & Vapnik, 1995) learns, for each genre, a weight per feature $w$ and a bias $b$, and scores a song
$x$ (its TF-IDF vector) with $f(x) = w \cdot x + b$. Training looks for weights that put songs of the genre on the
positive side and all other songs on the negative side **with a margin**: it minimises

$$\frac{1}{2}\lVert w \rVert^2 + C \sum_i c_{y_i}\, \max\big(0,\; 1 - y_i (w \cdot x_i + b)\big)$$

- The second term is the *hinge loss*: a song costs nothing if it is on the right side by at least 1, and costs more the
  further it is on the wrong side.
- The first term keeps the weights small. $C$ sets the balance: a **small $C$** (we use 0.1) prefers small weights over
  fitting every training song, which is stronger regularisation and helps with 365,000 sparse features.
- With six genres, `LinearSVC` trains one such model per genre ("one-vs-rest") and predicts the genre with the highest score.
- `class_weight="balanced"` sets $c_k = \frac{n}{K \cdot n_k}$ for a genre with $n_k$ songs out of $n$, with $K$ genres.
  A country song (2.6% of the data) then weighs about 16 times more than a pop song (41%), so the model can no longer
  ignore the small genres. This raises recall on rare genres at some cost in precision, which is the right trade for
  macro F1.

### Annex B — Kruskal-Wallis test and epsilon squared (notebook 4, SQ1)

The Kruskal-Wallis test (Kruskal & Wallis, 1952) asks whether several groups come from the same distribution without
assuming normality. All $N$ values are ranked together; if genre did not matter, every genre's average rank $\bar R_j$
would be close to the overall average $(N+1)/2$. The statistic

$$H = \frac{12}{N(N+1)} \sum_j n_j \bar R_j^2 - 3(N+1)$$

grows as the average ranks move apart. With thousands of songs even tiny differences give $p \approx 0$, so we also report
**epsilon squared**, $\varepsilon^2 = H / (N - 1)$ (Tomczak & Tomczak, 2014): the share of the rank variation explained by
the grouping, from 0 to 1.

### Annex C — Mann-Whitney U and Cliff's delta (notebook 4, SQ1 and SQ2)

For two groups with $n_1$ and $n_2$ songs, the Mann-Whitney $U$ (Mann & Whitney, 1947) counts, over all $n_1 n_2$ pairs,
how often the song from group 1 has the higher score (ties count ½). **Cliff's delta** (Cliff, 1993) turns this into an
effect size:

$$\delta = P(X_1 > X_2) - P(X_1 < X_2) = \frac{2U}{n_1 n_2} - 1$$

$\delta = 0$ means neither group tends to be higher; $\delta = -0.3$ means a random song of group 1 is lower than a random
song of group 2 noticeably more often than the reverse. It uses only the order of the scores, so it is not distorted by
the skewed, bounded VADER scores. We use the usual labels: $|\delta| < 0.147$ negligible, $< 0.33$ small, $< 0.474$ medium,
otherwise large (Romano et al., 2006).

### Annex D — Holm correction for multiple tests (notebook 4, SQ1)

With 5 genres there are 10 pairs. Testing each at 5% makes at least one false alarm quite likely. Holm's method (Holm, 1979)
sorts the $m$ p-values from smallest to largest, multiplies the $i$-th one by $(m - i + 1)$, and then makes the adjusted
values non-decreasing. It keeps the chance of *any* false positive below 5% while being less strict than multiplying every
p-value by $m$ (Bonferroni).

### Annex E — Wilcoxon signed-rank test (notebook 4, SQ5)

For paired data, here the chorus and the verses **of the same song**, the Wilcoxon signed-rank test (Wilcoxon, 1945)
takes the differences $d_i$ = chorus − verse, drops zeros, ranks the absolute differences and sums the ranks of the
positive ones. If the chorus were not systematically different, positive and negative differences of each size would be
equally likely and that sum would sit near half the total. Because each song is compared with itself, differences between
artists, decades and topics cancel out.

### Annex F — Bootstrap confidence intervals and Spearman correlation (notebook 4, SQ1, SQ3, SQ4, SQ5)

A **percentile bootstrap** interval (Efron & Tibshirani, 1993) estimates how much a statistic would move with a different
sample of songs: draw as many songs as we have, *with replacement*, recompute the statistic (a median, or a correlation
where pairs are drawn together), repeat 500–1,000 times, and report the 2.5th and 97.5th percentiles of the results. It
needs no assumption about the shape of the distribution.

**Spearman's ρ** (Spearman, 1904) is the ordinary correlation computed on ranks. It measures whether one variable tends to
go up when the other does (a monotonic relation, not necessarily a straight line), which suits views that are extremely
skewed and sentiment scores that are bounded.

### Annex G — Cohen's kappa (notebook 4, method check)

Raw agreement between two raters (here VADER and TextBlob, each labelling songs positive / neutral / negative) looks high
when one label dominates, because they would often agree by chance. Cohen's kappa (Cohen, 1960) corrects for that:

$$\kappa = \frac{p_o - p_e}{1 - p_e}$$

where $p_o$ is the observed agreement and $p_e$ the agreement expected if both raters assigned labels at random with their
own label frequencies. $\kappa = 0$ is chance level and $\kappa = 1$ is perfect agreement.